In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/binary-classification-with-a-bank-dataset-clone/sample_submission.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [3]:
train_df = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv")
test_df = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv")

print(train_df.shape)
print(test_df.shape)

(750000, 18)
(250000, 17)


In [4]:
train_df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [5]:
train_df.describe()

,id,age,balance,day,duration,campaign,pdays,previous,y
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,40.926395,1204.067397,16.117209,256.229144,2.577008,22.412733,0.298545,0.120651
std,216506.495284,10.098829,2836.096759,8.250832,272.555662,2.718514,77.319998,1.335926,0.325721
min,0.000000,18.000000,-8019.000000,1.000000,1.000000,1.000000,-1.000000,0.000000,0.000000
25%,187499.750000,33.000000,0.000000,9.000000,91.000000,1.000000,-1.000000,0.000000,0.000000
50%,374999.500000,39.000000,634.000000,17.000000,133.000000,2.000000,-1.000000,0.000000,0.000000
75%,562499.250000,48.000000,1390.000000,21.000000,361.000000,3.000000,-1.000000,0.000000,0.000000
max,749999.000000,95.000000,99717.000000,31.000000,4918.000000,63.000000,871.000000,200.000000,1.000000


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


In [7]:
train_df.isnull().sum()

id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [8]:
train_df['y'].value_counts()

y
0    659512
1     90488
Name: count, dtype: int64

Data Preprocessing

In [9]:
X = train_df.drop('y', axis =1)
y = train_df['y']

Handling Missing Values

In [10]:
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna(X[col].mode()[0])
        test_df[col] = test_df[col].fillna(test_df[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())
        test_df[col] = test_df[col].fillna(test_df[col].median())

Encoding

In [11]:
cat_cols = X.select_dtypes(include='object').columns

In [12]:
le = LabelEncoder()

for col in cat_cols:
    X[col] = le.fit_transform(X[col])
    test_df[col] = le.transform(test_df[col])

In [13]:
X.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,0,42,9,1,1,0,7,0,0,0,25,1,117,3,-1,0,3
1,1,38,1,1,1,0,514,0,0,2,18,6,185,1,-1,0,3
2,2,36,1,1,1,0,602,1,0,2,14,8,111,2,-1,0,3
3,3,27,8,2,1,0,34,1,0,2,28,8,10,2,-1,0,3
4,4,26,9,1,1,0,889,1,0,0,3,3,902,1,-1,0,3


Scaling

In [14]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(test_df)

Logistic Regression

In [15]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y)

LogisticRegression(max_iter=1000)

In [16]:
lr_preds = lr.predict(X_test_scaled)

In [17]:
submission_lr = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': lr_preds
})

submission_lr.to_csv("submission_logistic.csv", index=False)

Random Forest

In [18]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [19]:
rf_preds = rf.predict(test_df)

In [20]:
submission_rf = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': rf_preds
})

submission_rf.to_csv("submission_rf.csv", index=False)

In [21]:
rf2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf2.fit(X, y)

RandomForestClassifier(max_depth=20, min_samples_split=5, n_estimators=500,
                       n_jobs=-1, random_state=42)

In [22]:
rf2_preds = rf2.predict(test_df)

In [23]:
submission_rf2 = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': rf2_preds
})

submission_rf2.to_csv("submission_rf2.csv", index=False)

In [24]:
import xgboost as xgb

XGBoost

In [25]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, ...)

In [26]:
xgb_preds = xgb_model.predict(test_df)

In [27]:
submission_xgb = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': xgb_preds
})

submission_xgb.to_csv("submission_xgboost.csv", index=False)

In [28]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [29]:
xgb_balanced = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_balanced.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=-1,
              num_parallel_tree=None, ...)

In [30]:
xgb_bal_preds = xgb_balanced.predict(test_df)

In [31]:
submission_xgb_bal = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': xgb_bal_preds
})

submission_xgb_bal.to_csv("submission_xgboost_balanced.csv", index=False)

In [32]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

et.fit(X, y)

ExtraTreesClassifier(max_depth=20, min_samples_split=5, n_estimators=500,
                     n_jobs=-1, random_state=42)

In [33]:
et_preds = et.predict(test_df)

In [34]:
submission_et = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': et_preds
})

submission_et.to_csv("submission_extratrees.csv", index=False)

LightGBM

In [35]:
import lightgbm as lgb

pos_weight = (y == 0).sum() / (y == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X, y)

[LightGBM] [Info] Number of positive: 90488, number of negative: 659512
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038929 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1256
[LightGBM] [Info] Number of data points in the train set: 750000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120651 -> initscore=-1.986283
[LightGBM] [Info] Start training from score -1.986283
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=8,
               n_estimators=800, n_jobs=-1, num_leaves=64, random_state=42,
               scale_pos_weight=np.float64(7.288391830961012), subsample=0.8)

In [36]:
lgb_preds = lgb_model.predict(test_df)

In [37]:
submission_lgb = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': lgb_preds
})

submission_lgb.to_csv("submission_lightgbm.csv", index=False)

CatBoost

In [38]:
from catboost import CatBoostClassifier
cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=0
)

cat_model.fit(X, y)

In [39]:
cat_preds = cat_model.predict(test_df)

In [40]:
submission_cat = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': cat_preds.astype(int)
})

submission_cat.to_csv("submission_catboost.csv", index=False)

In [41]:
xgb_prob = xgb_balanced.predict_proba(X)[:, 1]
lgb_prob = lgb_model.predict_proba(X)[:, 1]

stack_X = np.column_stack((xgb_prob, lgb_prob))

In [42]:
meta = LogisticRegression()
meta.fit(stack_X, y)

LogisticRegression()

In [43]:
xgb_test_prob = xgb_balanced.predict_proba(test_df)[:, 1]
lgb_test_prob = lgb_model.predict_proba(test_df)[:, 1]

stack_test = np.column_stack((xgb_test_prob, lgb_test_prob))

In [44]:
stack_preds = (meta.predict_proba(stack_test)[:, 1] > 0.35).astype(int)

In [46]:
submission_stack = pd.DataFrame({
    'id': np.arange(750000, 750000 + len(test_df)),
    'y': stack_preds
})

submission_stack.to_csv("submission_stacked.csv", index=False)